# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prarthanamahesh21-hub/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
from datasets import load_dataset
import duckdb
import pandas as pd
import numpy as np

# Load Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", HF_TOKEN is not None)

# Load datasets
dim_content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    token=HF_TOKEN
)

fact_query = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d",
    token=HF_TOKEN
)

fact_performance = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=HF_TOKEN
)

# Convert to train tables
content = dim_content["train"]
query_90d = fact_query["train"]
performance = fact_performance["train"]

# Create DuckDB connection
con = duckdb.connect()

# Register tables
con.register("dim_content", content._data.table)
con.register("fact_content_query_90d", query_90d._data.table)
con.register(
    "fact_content_daily_performance",
    performance._data.table
)

print("Setup complete.")
print("Content rows:", len(content))
print("Query rows:", len(query_90d))
print("Performance rows:", len(performance))

HF token loaded: True


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Setup complete.
Content rows: 519606
Query rows: 2414248
Performance rows: 78835655


In [2]:
feature_cols = [
    "content_age_days",
    "word_count",
    "gsc_impressions_90d",
    "gsc_clicks_90d",
    "ga4_pageviews_90d"
]

features_march = con.sql("""
WITH performance_90d AS (
    SELECT
        content_hash_id,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS gsc_impressions_90d,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS gsc_clicks_90d,

        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN COALESCE(ga4_pageviews, 0)
                ELSE 0
            END
        ) AS ga4_pageviews_90d

    FROM fact_content_daily_performance

    WHERE report_date >= DATE '2025-12-01'
      AND report_date < DATE '2026-03-01'

    GROUP BY content_hash_id
)

SELECT
    c.content_hash_id,

    DATE_DIFF(
        'day',
        CAST(c.content_created_date AS DATE),
        DATE '2026-03-01'
    ) AS content_age_days,

    c.word_count,
    p.gsc_impressions_90d,
    p.gsc_clicks_90d,
    p.ga4_pageviews_90d

FROM dim_content c

LEFT JOIN performance_90d p
    ON c.content_hash_id = p.content_hash_id

WHERE c.content_created_date < DATE '2026-03-01'
  AND c.is_published IS TRUE
  AND c.is_deleted IS FALSE
""").df()

print("March feature frame:", features_march.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March feature frame: (303321, 6)


In [3]:
future_label = con.sql("""
SELECT
    content_hash_id,

    SUM(
        CASE
            WHEN gsc_data_available IS TRUE
            THEN COALESCE(gsc_clicks, 0)
            ELSE 0
        END
    ) AS future_gsc_clicks

FROM fact_content_daily_performance

WHERE report_date >= DATE '2026-04-01'
  AND report_date < DATE '2026-05-01'

GROUP BY content_hash_id
""").df()

future_label["label"] = (
    future_label["future_gsc_clicks"] > 0
).astype(int)

print("Future label rows:", len(future_label))
print("Positive labels:", future_label["label"].sum())
print("Negative labels:", (future_label["label"] == 0).sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Future label rows: 362172
Positive labels: 67832
Negative labels: 294340


In [4]:
import duckdb

content = dim_content["train"]
query_90d = fact_query["train"]

con = duckdb.connect()

con.register("dim_content", content._data.table)
con.register("fact_content_query_90d", query_90d._data.table)

print("DuckDB setup complete")

DuckDB setup complete


In [5]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

from datasets import load_dataset

dim_content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    token=HF_TOKEN
)

fact_query = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d",
    token=HF_TOKEN
)

In [6]:
import duckdb

content = dim_content["train"]
query_90d = fact_query["train"]

con = duckdb.connect()

con.register("dim_content", content._data.table)
con.register("fact_content_query_90d", query_90d._data.table)

print("DuckDB setup complete")
print("Content rows:", len(content))
print("Query 90d rows:", len(query_90d))

DuckDB setup complete
Content rows: 519606
Query 90d rows: 2414248


In [7]:
fact_performance = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=HF_TOKEN
)

performance = fact_performance["train"]

con.register(
    "fact_content_daily_performance",
    performance._data.table
)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 1. Two paper findings + my methodology questions

### Finding 1 — The baseline rule performed below the base rate

The paper reports that its hand-written baseline rule achieved a Precision@50 of 0.500, while the base rate was 0.578 on the evaluated client-grouped test split. The result is useful because it shows that a simple rule based on stale content and visibility did not outperform the base rate in this evaluation.

**My methodology question:** Where exactly does the label come from, and does it represent the outcome that the baseline is intended to identify? I would want to verify that the positive label is constructed from a future outcome window that is separate from the feature window, and that the label is not derived from the same signals used by the rule.

This is a constructive question because the meaning of a measured precision depends on how the outcome was defined and whether it was available independently of the inputs.

### Finding 2 — Random validation substantially increased the measured score

The paper reports that the same random forest model achieved a Precision@50 of 0.900 with a naive random split, compared with 0.700 under a client-grouped split. The paper uses the grouped split to prevent pages from the same client appearing in both training and test data.

**My methodology question:** Does the validation design support the strength of the claim being made? In particular, if pages from the same client can appear in both training and test sets, the model may benefit from client-specific patterns that would not be available when evaluating on unseen clients. I would therefore want the validation design to match the intended generalization setting before interpreting the higher score as evidence of model performance.

This question does not reject the finding. It asks whether the evaluation design provides evidence for the scope of the claim.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

The Week-5 evaluation used a random stratified 80/20 split. For this audit, I use a client-grouped split so that pages from the same client cannot appear in both the training and test sets.

This is a stricter validation design because the model is evaluated on clients that were not represented in training. I compare the Week-5 random-split result with the client-grouped result to see how the measured performance changes under a more conservative evaluation.

The comparison is directional. The purpose is to understand how validation design affects the measured result rather than to claim that one score represents performance in every future setting.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [13]:
# Rebuild the Week-5 modeling frame for the Week-6 audit

feature_cols = [
    "content_age_days",
    "word_count",
    "gsc_impressions_90d",
    "gsc_clicks_90d",
    "ga4_pageviews_90d"
]

# 1. Build March feature frame
features_march = con.sql("""
WITH performance_90d AS (
    SELECT
        content_hash_id,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS gsc_impressions_90d,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS gsc_clicks_90d,

        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN COALESCE(ga4_pageviews, 0)
                ELSE 0
            END
        ) AS ga4_pageviews_90d

    FROM fact_content_daily_performance

    WHERE report_date >= DATE '2025-12-01'
      AND report_date < DATE '2026-03-01'

    GROUP BY content_hash_id
)

SELECT
    c.content_hash_id,

    DATE_DIFF(
        'day',
        CAST(c.content_created_date AS DATE),
        DATE '2026-03-01'
    ) AS content_age_days,

    c.word_count,
    p.gsc_impressions_90d,
    p.gsc_clicks_90d,
    p.ga4_pageviews_90d

FROM dim_content c

LEFT JOIN performance_90d p
    ON c.content_hash_id = p.content_hash_id

WHERE c.content_created_date < DATE '2026-03-01'
  AND c.is_published IS TRUE
  AND c.is_deleted IS FALSE
""").df()

print("March feature frame:", features_march.shape)


# 2. Build April future label
future_label = con.sql("""
SELECT
    content_hash_id,

    SUM(
        CASE
            WHEN gsc_data_available IS TRUE
            THEN COALESCE(gsc_clicks, 0)
            ELSE 0
        END
    ) AS future_gsc_clicks

FROM fact_content_daily_performance

WHERE report_date >= DATE '2026-04-01'
  AND report_date < DATE '2026-05-01'

GROUP BY content_hash_id
""").df()

future_label["label"] = (
    future_label["future_gsc_clicks"] > 0
).astype(int)

print("Future label rows:", len(future_label))
print("Positive labels:", future_label["label"].sum())
print("Negative labels:", (future_label["label"] == 0).sum())


# 3. Create model frame
model_df = features_march.merge(
    future_label[["content_hash_id", "label"]],
    on="content_hash_id",
    how="inner"
)

print("Model frame shape:", model_df.shape)


# 4. Add client identifier
client_map = con.sql("""
SELECT
    content_hash_id,
    client_hash_id
FROM dim_content
""").df()

model_df_audit = model_df.merge(
    client_map,
    on="content_hash_id",
    how="left"
)

print("Model audit frame:", model_df_audit.shape)
print("Unique clients:", model_df_audit["client_hash_id"].nunique())
print(
    "Missing client IDs:",
    model_df_audit["client_hash_id"].isna().sum()
)


# 5. Client-grouped split
from sklearn.model_selection import GroupShuffleSplit

X = model_df_audit[feature_cols]
y = model_df_audit["label"]
groups = model_df_audit["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_grouped = X.iloc[train_idx].copy()
X_test_grouped = X.iloc[test_idx].copy()

y_train_grouped = y.iloc[train_idx].copy()
y_test_grouped = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Training rows:", len(X_train_grouped))
print("Test rows:", len(X_test_grouped))
print("Training clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

print(
    "Clients shared between train/test:",
    len(set(groups_train) & set(groups_test))
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March feature frame: (303321, 6)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Future label rows: 362172
Positive labels: 67832
Negative labels: 294340
Model frame shape: (293504, 7)
Model audit frame: (293504, 8)
Unique clients: 50
Missing client IDs: 0
Training rows: 251424
Test rows: 42080
Training clients: 40
Test clients: 10
Clients shared between train/test: 0


In [14]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Train Logistic Regression on client-grouped training data

grouped_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

grouped_model.fit(
    X_train_grouped,
    y_train_grouped
)

# Predict probabilities on unseen clients
grouped_scores = grouped_model.predict_proba(
    X_test_grouped
)[:, 1]

grouped_auc = roc_auc_score(
    y_test_grouped,
    grouped_scores
)

print(
    "Client-grouped Logistic Regression ROC-AUC:",
    round(grouped_auc, 4)
)

Client-grouped Logistic Regression ROC-AUC: 0.8814


In [15]:
from sklearn.model_selection import train_test_split

# Recreate the original Week-5 random stratified split

X_random = model_df[feature_cols]
y_random = model_df["label"]

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X_random,
    y_random,
    test_size=0.20,
    random_state=42,
    stratify=y_random
)

random_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

random_model.fit(
    X_train_random,
    y_train_random
)

random_scores = random_model.predict_proba(
    X_test_random
)[:, 1]

random_auc = roc_auc_score(
    y_test_random,
    random_scores
)

print(
    "Week-5 random-split Logistic Regression ROC-AUC:",
    round(random_auc, 4)
)

Week-5 random-split Logistic Regression ROC-AUC: 0.8995


In [16]:
validation_comparison = pd.DataFrame({
    "evaluation": [
        "Week-5 random stratified split",
        "Week-6 client-grouped split"
    ],
    "roc_auc": [
        random_auc,
        grouped_auc
    ]
})

validation_comparison["roc_auc"] = (
    validation_comparison["roc_auc"].round(4)
)

print(validation_comparison.to_string(index=False))

print(
    "\nChange in ROC-AUC:",
    round(grouped_auc - random_auc, 4)
)

                    evaluation  roc_auc
Week-5 random stratified split   0.8995
   Week-6 client-grouped split   0.8814

Change in ROC-AUC: -0.018


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Before/after interpretation

The Logistic Regression model measured a ROC-AUC of **0.8995** under the Week-5 random stratified split and **0.8814** under the client-grouped split. The measured difference was **−0.0180 ROC-AUC**.

The lower score under client-grouped validation suggests that the random split may have provided a somewhat easier evaluation setting. The grouped result still shows measured predictive discrimination on clients not represented in training, but the difference means the Week-5 result should not be treated as a universal estimate of performance.

This is a directional validation finding. The client-grouped result provides stronger evidence for generalization across clients and is more appropriate for cautious decision-support interpretation.


In [17]:
# Leakage audit on final feature set

print("Final features:")
for feature in feature_cols:
    print("-", feature)

print("\nTarget column:", "label")

# Check that target is not included as a feature
print(
    "\nLabel included in features:",
    "label" in feature_cols
)

print(
    "Future outcome included in features:",
    "future_gsc_clicks" in feature_cols
)

# Check for target-like columns
target_like = [
    col for col in model_df_audit.columns
    if any(
        word in col.lower()
        for word in [
            "label",
            "target",
            "future",
            "outcome"
        ]
    )
]

print("\nTarget/outcome-like columns in model frame:")
print(target_like)

# Feature overlap with target/outcome columns
overlap = set(feature_cols) & {
    "label",
    "future_gsc_clicks"
}

print("\nFeature/target overlap:", overlap)

# Check missing values
print("\nMissing values in final features:")
print(model_df_audit[feature_cols].isna().sum())

# Check duplicate page IDs
print(
    "\nDuplicate content IDs:",
    model_df_audit["content_hash_id"].duplicated().sum()
)

Final features:
- content_age_days
- word_count
- gsc_impressions_90d
- gsc_clicks_90d
- ga4_pageviews_90d

Target column: label

Label included in features: False
Future outcome included in features: False

Target/outcome-like columns in model frame:
['label']

Feature/target overlap: set()

Missing values in final features:
content_age_days            0
word_count             105661
gsc_impressions_90d      2863
gsc_clicks_90d           2863
ga4_pageviews_90d        2863
dtype: int64

Duplicate content IDs: 0


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Original claim

The Week-5 model achieved a ROC-AUC of 0.8995 compared with the Week-4 baseline, showing that the model can identify which pages should be refreshed.

### Rewritten claim

On the evaluated data, the Logistic Regression model measured a ROC-AUC of **0.8995** under the Week-5 random stratified split and **0.8814** under the client-grouped split.

The client-grouped result provides a more conservative measurement of predictive discrimination across unseen clients. The results are **directional** and can support **decision-support** for prioritizing pages for human review.

These measurements do not establish that the model determines which pages should be refreshed or that using the model causes improved content performance.


In [18]:
# Summarize the measured results supporting the rewritten claim

claim_audit = pd.DataFrame({
    "evaluation": [
        "Week-5 random stratified split",
        "Week-6 client-grouped split"
    ],
    "roc_auc": [
        random_auc,
        grouped_auc
    ]
})

claim_audit["roc_auc"] = claim_audit["roc_auc"].round(4)

print("Measured validation results:")
print(claim_audit.to_string(index=False))

print(
    "\nMeasured change from random to grouped split:",
    round(grouped_auc - random_auc, 4)
)

print(
    "\nLeakage check:",
    "No direct feature/target overlap identified."
)

Measured validation results:
                    evaluation  roc_auc
Week-5 random stratified split   0.8995
   Week-6 client-grouped split   0.8814

Measured change from random to grouped split: -0.018

Leakage check: No direct feature/target overlap identified.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [19]:
# Error analysis on the client-grouped test set

grouped_error_df = X_test_grouped.copy()

grouped_error_df["actual"] = y_test_grouped.values
grouped_error_df["predicted_score"] = grouped_scores

grouped_error_df["predicted_class"] = (
    grouped_scores >= 0.5
).astype(int)

grouped_error_df["error_type"] = np.select(
    [
        (grouped_error_df["actual"] == 1) &
        (grouped_error_df["predicted_class"] == 1),

        (grouped_error_df["actual"] == 0) &
        (grouped_error_df["predicted_class"] == 0),

        (grouped_error_df["actual"] == 0) &
        (grouped_error_df["predicted_class"] == 1),

        (grouped_error_df["actual"] == 1) &
        (grouped_error_df["predicted_class"] == 0)
    ],
    [
        "True Positive",
        "True Negative",
        "False Positive",
        "False Negative"
    ],
    default="Unknown"
)

print("Prediction outcome counts:")
print(grouped_error_df["error_type"].value_counts())

print("\nHighest-confidence false positives:")
print(
    grouped_error_df[
        grouped_error_df["error_type"] == "False Positive"
    ]
    .sort_values("predicted_score", ascending=False)
    .head(5)
    .to_string(index=False)
)

print("\nHighest-confidence false negatives:")
print(
    grouped_error_df[
        grouped_error_df["error_type"] == "False Negative"
    ]
    .sort_values("predicted_score", ascending=True)
    .head(5)
    .to_string(index=False)
)

Prediction outcome counts:
error_type
True Negative     31988
True Positive      4737
False Negative     4463
False Positive      892
Name: count, dtype: int64

Highest-confidence false positives:
 content_age_days  word_count  gsc_impressions_90d  gsc_clicks_90d  ga4_pageviews_90d  actual  predicted_score  predicted_class     error_type
              189        1409               4796.0           212.0                0.0       0              1.0                1 False Positive
              290        2585             116593.0            74.0               88.0       0              1.0                1 False Positive
              130        2850              24739.0           137.0              171.0       0              1.0                1 False Positive
              130        1501              20356.0           187.0              236.0       0              1.0                1 False Positive
              338        2631              14001.0           147.0              186.0   

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.